In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

# =====================================================
# DATASET
# =====================================================

class FaultDataset(torch.utils.data.Dataset):
    def __init__(self, root):
        self.s = os.path.join(root, "seis")
        self.f = os.path.join(root, "fault")
        self.files = sorted(os.listdir(self.s))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        name = self.files[i]

        seis = np.fromfile(
            os.path.join(self.s, name),
            dtype=np.float32
        )

        fault = np.fromfile(
            os.path.join(self.f, name),
            dtype=np.float32
        )

        size = int(round(seis.shape[0] ** (1/3)))

        seis = seis.reshape(size, size, size)
        fault = fault.reshape(size, size, size)

        seis = (seis - seis.mean()) / (seis.std() + 1e-6)
        fault = (fault > 0).astype(np.float32)

        return (
            torch.tensor(seis).unsqueeze(0).float(),
            torch.tensor(fault).unsqueeze(0).float()
        )


# =====================================================
# MODEL
# =====================================================

class DoubleConv(nn.Module):

    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.conv1 = nn.Conv3d(
            in_ch,
            out_ch,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv3d(
            out_ch,
            out_ch,
            kernel_size=3,
            padding=1
        )

    def forward(self, x):

        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))

        return x


class FaultSeg3D(nn.Module):

    def __init__(self):
        super().__init__()

        self.pool = nn.MaxPool3d(2)

        self.conv1 = DoubleConv(1, 16)
        self.conv2 = DoubleConv(16, 32)
        self.conv3 = DoubleConv(32, 64)

        self.conv4 = DoubleConv(64, 512)

        self.up1 = nn.Upsample(
            scale_factor=2,
            mode="nearest"
        )

        self.dec1 = DoubleConv(576, 64)

        self.up2 = nn.Upsample(
            scale_factor=2,
            mode="nearest"
        )

        self.dec2 = DoubleConv(96, 32)

        self.up3 = nn.Upsample(
            scale_factor=2,
            mode="nearest"
        )

        self.dec3 = DoubleConv(48, 16)

        self.final = nn.Conv3d(
            16,
            1,
            kernel_size=1
        )

    def forward(self, x):

        c1 = self.conv1(x)

        c2 = self.conv2(
            self.pool(c1)
        )

        c3 = self.conv3(
            self.pool(c2)
        )

        c4 = self.conv4(
            self.pool(c3)
        )

        x = self.up1(c4)
        x = torch.cat([x, c3], dim=1)
        x = self.dec1(x)

        x = self.up2(x)
        x = torch.cat([x, c2], dim=1)
        x = self.dec2(x)

        x = self.up3(x)
        x = torch.cat([x, c1], dim=1)
        x = self.dec3(x)

        return self.final(x)
model = FaultSeg3D()



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.load_state_dict(
    torch.load(
        "../checkpoints/fault/faultseg3d_best.pth",
        map_location=device
    )
)
model.eval()
loaded_model = model
#loaded_model.load_weights("pretrained_model.hdf5")
print("Loaded model from disk")

# training image dimensions
n1, n2, n3 = 128, 128, 128

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PatchEmbed3D(nn.Module):
    def __init__(self, in_ch, embed_dim, patch_size=2):
        super().__init__()
        self.proj = nn.Conv3d(in_ch, embed_dim, patch_size, patch_size)

    def forward(self, x):
        return self.proj(x)

class TransformerBlock(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()

        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(
            dim,
            heads,
            batch_first=True
        )

        self.norm2 = nn.LayerNorm(dim)

        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim)
        )

    def forward(self, x):
        x = x + self.attn(
            self.norm1(x),
            self.norm1(x),
            self.norm1(x)
        )[0]

        x = x + self.mlp(self.norm2(x))
        return x

class TransformerBottleneck3D(nn.Module):
    def __init__(
        self,
        in_ch=64,
        embed_dim=64,
        patch_size=2,
        depth=2
    ):
        super().__init__()

        self.patch = PatchEmbed3D(
            in_ch,
            embed_dim,
            patch_size
        )

        self.transformer = nn.Sequential(
            *[
                TransformerBlock(embed_dim)
                for _ in range(depth)
            ]
        )

    def forward(self, x):

        B,C,D,H,W = x.shape

        x = self.patch(x)

        B,C,Dd,Hd,Wd = x.shape

        x = x.flatten(2).transpose(1,2)
        x = self.transformer(x)
        x = x.transpose(1,2).view(
            B,C,Dd,Hd,Wd
        )

        x = F.interpolate(
            x,
            size=(D,H,W),
            mode="trilinear",
            align_corners=False
        )

        return x

class Generator(nn.Module):

    def __init__(self):
        super().__init__()

        self.pool = nn.MaxPool3d(2)

        self.e1 = self.block(1,16)
        self.e2 = self.block(16,32)
        self.e3 = self.block(32,64)

        self.bottleneck = TransformerBottleneck3D(
            64,
            64
        )

        self.up3 = nn.ConvTranspose3d(
            64,64,2,2
        )
        self.d3 = self.block(128,64)

        self.up2 = nn.ConvTranspose3d(
            64,32,2,2
        )
        self.d2 = self.block(64,32)

        self.up1 = nn.ConvTranspose3d(
            32,16,2,2
        )
        self.d1 = self.block(32,16)

        self.final = nn.Conv3d(
            16,1,1
        )

    def block(self,in_ch,out_ch):
        return nn.Sequential(
            nn.Conv3d(
                in_ch,
                out_ch,
                3,
                padding=1
            ),
            nn.InstanceNorm3d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv3d(
                out_ch,
                out_ch,
                3,
                padding=1
            ),
            nn.InstanceNorm3d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self,x):

        x1 = self.e1(x)
        x2 = self.e2(self.pool(x1))
        x3 = self.e3(self.pool(x2))

        x4 = self.bottleneck(
            self.pool(x3)
        )

        x = self.up3(x4)
        x = self.d3(
            torch.cat([x,x3],dim=1)
        )

        x = self.up2(x)
        x = self.d2(
            torch.cat([x,x2],dim=1)
        )

        x = self.up1(x)
        x = self.d1(
            torch.cat([x,x1],dim=1)
        )

        return self.final(x)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = Generator().to(device)

model.load_state_dict(
    torch.load(
        "../checkpoints/H07.pth",
        map_location=device
    )
)

model.eval()

print("FaultGAN loaded")

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

# ==========================================================
# SAME INPUT PATCH
# ==========================================================
gx = np.fromfile(
    "../data/train-20260410T053600Z-3-001/train/seis/8.dat",
    dtype=np.float32
).reshape(128,128,128)

# ==========================================================
# FAULTSEG
# ==========================================================
gx_fs = gx.copy()
gx_fs = gx_fs - gx_fs.min()
gx_fs = gx_fs / (gx_fs.max() + 1e-8)
gx_fs = gx_fs * 255.0

x_fs = gx_fs.reshape(1,1,128,128,128)

loaded_model.eval()

x_fs = torch.tensor(x_fs, dtype=torch.float32)

with torch.no_grad():
    output = loaded_model(x_fs)

faultseg_prob = output.squeeze().cpu().numpy()
faultseg_mask = (faultseg_prob > 0.5).astype(np.uint8)

# ==========================================================
# FAULTGAN
# ==========================================================
gx_fg = gx.copy()
gx_fg = (gx_fg - gx_fg.mean())/(gx_fg.std()+1e-6)

device = next(model.parameters()).device

x_fg = (
    torch.tensor(gx_fg)
    .float()
    .unsqueeze(0)
    .unsqueeze(0)
    .to(device)
)

with torch.no_grad():
    logits = model(x_fg)
    faultgan_prob = torch.sigmoid(logits)

faultgan_mask = (
    faultgan_prob.squeeze()
    .cpu()
    .numpy() > 0.5
).astype(np.uint8)

# ==========================================================
# BINARY ONLY
# ==========================================================
k1 = 50
k2 = 110
k3 = 60

plt.figure(figsize=(18,12))

# INLINE
plt.subplot(3,3,1)
plt.imshow(gx[k1,:,:].T,cmap="gray")
plt.title("Seismic Inline")

plt.subplot(3,3,2)
plt.imshow(faultseg_mask[k1,:,:].T,cmap="gray")
plt.title("FaultSeg Binary")

plt.subplot(3,3,3)
plt.imshow(faultgan_mask[k1,:,:].T,cmap="gray")
plt.title("FaultGAN Binary")

# XLINE
plt.subplot(3,3,4)
plt.imshow(gx[:,k2,:].T,cmap="gray")
plt.title("Seismic Xline")

plt.subplot(3,3,5)
plt.imshow(faultseg_mask[:,k2,:].T,cmap="gray")
plt.title("FaultSeg Binary")

plt.subplot(3,3,6)
plt.imshow(faultgan_mask[:,k2,:].T,cmap="gray")
plt.title("FaultGAN Binary")

# TIME
plt.subplot(3,3,7)
plt.imshow(gx[:,:,k3],cmap="gray")
plt.title("Seismic Time")

plt.subplot(3,3,8)
plt.imshow(faultseg_mask[:,:,k3],cmap="gray")
plt.title("FaultSeg Binary")

plt.subplot(3,3,9)
plt.imshow(faultgan_mask[:,:,k3],cmap="gray")
plt.title("FaultGAN Binary")

plt.tight_layout()
plt.show()

In [ ]:
import torch

print(torch.__version__)
print(torch.version.cuda)

print(torch.backends.cudnn.is_available())
print(torch.backends.cudnn.version())

print(torch.cuda.get_device_name(0))

In [ ]:
!nvidia-smi

In [ ]:
def compare_faultseg_faultgan(x, faultseg_mask, faultgan_mask, title=""):

    # reshape exactly like your code
    x = np.transpose(x, (0,2,3,4,1))
    faultseg_mask = np.transpose(faultseg_mask, (0,2,3,4,1))
    faultgan_mask = np.transpose(faultgan_mask, (0,2,3,4,1))

    k1 = 50
    k2 = 110
    k3 = 60

    plt.figure(figsize=(18,8))
    plt.suptitle(title)

    # =====================================================
    # ROW 1 : FAULTSEG
    # =====================================================

    plt.subplot(2,6,1)
    plt.imshow(np.transpose(x[0,k1,:,:,0]), cmap='bone', aspect=1)
    plt.title("Input Inline")

    plt.subplot(2,6,2)
    plt.imshow(np.transpose(faultseg_mask[0,k1,:,:,0]), cmap='gray', aspect=1)
    plt.title("FaultSeg Inline")

    plt.subplot(2,6,3)
    plt.imshow(np.transpose(x[0,:,k2,:,0]), cmap='bone', aspect=1)
    plt.title("Input Xline")

    plt.subplot(2,6,4)
    plt.imshow(np.transpose(faultseg_mask[0,:,k2,:,0]), cmap='gray', aspect=1)
    plt.title("FaultSeg Xline")

    plt.subplot(2,6,5)
    plt.imshow(np.transpose(x[0,:,:,k3,0]), cmap='bone', aspect=1)
    plt.title("Input Time")

    plt.subplot(2,6,6)
    plt.imshow(np.transpose(faultseg_mask[0,:,:,k3,0]), cmap='gray', aspect=1)
    plt.title("FaultSeg Time")

    # =====================================================
    # ROW 2 : FAULTGAN
    # =====================================================

    plt.subplot(2,6,7)
    plt.imshow(np.transpose(x[0,k1,:,:,0]), cmap='bone', aspect=1)
    plt.title("Input Inline")

    plt.subplot(2,6,8)
    plt.imshow(np.transpose(faultgan_mask[0,k1,:,:,0]), cmap='gray', aspect=1)
    plt.title("FaultGAN Inline")

    plt.subplot(2,6,9)
    plt.imshow(np.transpose(x[0,:,k2,:,0]), cmap='bone', aspect=1)
    plt.title("Input Xline")

    plt.subplot(2,6,10)
    plt.imshow(np.transpose(faultgan_mask[0,:,k2,:,0]), cmap='gray', aspect=1)
    plt.title("FaultGAN Xline")

    plt.subplot(2,6,11)
    plt.imshow(np.transpose(x[0,:,:,k3,0]), cmap='bone', aspect=1)
    plt.title("Input Time")

    plt.subplot(2,6,12)
    plt.imshow(np.transpose(faultgan_mask[0,:,:,k3,0]), cmap='gray', aspect=1)
    plt.title("FaultGAN Time")

    plt.tight_layout()
    plt.show()

In [ ]:
# ==========================================================
# LOAD SAME PATCH
# ==========================================================
gx = np.fromfile(
    "../data/train-20260410T053600Z-3-001/train/seis/8.dat",
    dtype=np.float32
).reshape(128,128,128)

# ----------------------------------------------------------
# FAULTSEG INPUT
# ----------------------------------------------------------
gx_fs = gx.copy()
gx_fs = gx_fs - gx_fs.min()
gx_fs = gx_fs / (gx_fs.max() + 1e-8)
gx_fs = gx_fs * 255

x_fs = gx_fs.reshape(1,128,128,128,1)
loaded_model.eval()

with torch.no_grad():

    x_torch = torch.tensor(
        x_fs,
        dtype=torch.float32
    )

    # NDHWC -> NCDHW
    x_torch = x_torch.permute(
        0, 4, 1, 2, 3
    )

    x_torch = x_torch.cuda()

    logits = loaded_model(x_torch)

    faultseg_prob = torch.sigmoid(
        logits
    ).cpu().numpy()
faultseg_prob = faultseg_prob[0,0]

faultseg_mask = (
    faultseg_prob > 0.5
).astype(np.float32)
faultseg_mask = (faultseg_prob > 0.5).astype(np.float32)

# ----------------------------------------------------------
# FAULTGAN INPUT
# ----------------------------------------------------------
gx_fg = gx.copy()
gx_fg = (gx_fg - gx_fg.mean())/(gx_fg.std()+1e-6)

x_fg = torch.tensor(
    gx_fg,
    dtype=torch.float32
).unsqueeze(0).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(x_fg)
    faultgan_prob = torch.sigmoid(logits)

faultgan_prob = faultgan_prob.squeeze().cpu().numpy()
faultgan_mask = (faultgan_prob > 0.5).astype(np.float32)

# ----------------------------------------------------------
# CREATE x VARIABLE FOR VISUALIZATION FUNCTION
# ----------------------------------------------------------
x = x_fg.cpu().numpy()

# ----------------------------------------------------------
# CALL PLOT
# ----------------------------------------------------------
compare_faultseg_faultgan(
    x,
    faultseg_mask[np.newaxis,np.newaxis,:,:,:],
    faultgan_mask[np.newaxis,np.newaxis,:,:,:],
    title="FaultSeg vs FaultGAN Binary Comparison"
)

In [ ]:
# -----------------------------
# INFERENCE ON PATCH
# -----------------------------
def infer_both_models(patch):

    # =====================================================
    # FAULTSEG
    # =====================================================
    gx_fs = patch.copy()

    gx_fs = gx_fs - gx_fs.min()
    gx_fs = gx_fs / (gx_fs.max() + 1e-8)
    gx_fs = gx_fs * 255

    x_fs = gx_fs.reshape(1,128,128,128,1)

    faultseg_prob = loaded_model.predict(
        x_fs,
        verbose=0
    )[0,:,:,:,0]

    faultseg_mask = (faultseg_prob > 0.5).astype(np.float32)

    # =====================================================
    # FAULTGAN
    # =====================================================
    gx_fg = patch.copy()

    gx_fg = (gx_fg - gx_fg.mean())/(gx_fg.std()+1e-6)

    x_fg = torch.tensor(
        gx_fg
    ).unsqueeze(0).unsqueeze(0).float().to(device)

    with torch.no_grad():
        logits = model(x_fg)
        faultgan_prob = torch.sigmoid(logits)

    faultgan_prob = faultgan_prob.squeeze().cpu().numpy()

    faultgan_mask = (
        faultgan_prob > 0.5
    ).astype(np.float32)

    return (
        x_fg.cpu().numpy(),
        faultseg_mask,
        faultgan_mask
    )

In [ ]:
def compare_faultseg_faultgan(
    x,
    faultseg_mask,
    faultgan_mask,
    title=""
):

    x = np.transpose(x,(0,2,3,4,1))

    faultseg_mask = np.transpose(
        faultseg_mask[np.newaxis,np.newaxis,:,:,:],
        (0,2,3,4,1)
    )

    faultgan_mask = np.transpose(
        faultgan_mask[np.newaxis,np.newaxis,:,:,:],
        (0,2,3,4,1)
    )

    k1 = 50
    k2 = 110
    k3 = 60

    plt.figure(figsize=(18,8))
    plt.suptitle(title)

    # ==================================
    # FAULTSEG ROW
    # ==================================

    plt.subplot(2,6,1)
    plt.imshow(np.transpose(x[0,k1,:,:,0]), cmap='bone')
    plt.title("Input Inline")

    plt.subplot(2,6,2)
    plt.imshow(np.transpose(faultseg_mask[0,k1,:,:,0]), cmap='gray')
    plt.title("FaultSeg")

    plt.subplot(2,6,3)
    plt.imshow(np.transpose(x[0,:,k2,:,0]), cmap='bone')
    plt.title("Input Xline")

    plt.subplot(2,6,4)
    plt.imshow(np.transpose(faultseg_mask[0,:,k2,:,0]), cmap='gray')
    plt.title("FaultSeg")

    plt.subplot(2,6,5)
    plt.imshow(np.transpose(x[0,:,:,k3,0]), cmap='bone')
    plt.title("Input Time")

    plt.subplot(2,6,6)
    plt.imshow(np.transpose(faultseg_mask[0,:,:,k3,0]), cmap='gray')
    plt.title("FaultSeg")

    # ==================================
    # FAULTGAN ROW
    # ==================================

    plt.subplot(2,6,7)
    plt.imshow(np.transpose(x[0,k1,:,:,0]), cmap='bone')
    plt.title("Input Inline")

    plt.subplot(2,6,8)
    plt.imshow(np.transpose(faultgan_mask[0,k1,:,:,0]), cmap='gray')
    plt.title("FaultGAN")

    plt.subplot(2,6,9)
    plt.imshow(np.transpose(x[0,:,k2,:,0]), cmap='bone')
    plt.title("Input Xline")

    plt.subplot(2,6,10)
    plt.imshow(np.transpose(faultgan_mask[0,:,k2,:,0]), cmap='gray')
    plt.title("FaultGAN")

    plt.subplot(2,6,11)
    plt.imshow(np.transpose(x[0,:,:,k3,0]), cmap='bone')
    plt.title("Input Time")

    plt.subplot(2,6,12)
    plt.imshow(np.transpose(faultgan_mask[0,:,:,k3,0]), cmap='gray')
    plt.title("FaultGAN")

    plt.tight_layout()
    plt.show()

In [ ]:
volume = np.fromfile("../data/F3_block.npy", dtype=np.float32)

In [ ]:
print(type(volume))
print(volume.shape)

In [ ]:
def get_patch(volume):

    D,H,W = volume.shape

    cube_size = 128

    z = np.random.randint(0, D-cube_size+1)
    y = np.random.randint(0, H-cube_size+1)

    patch = volume[
        z:z+cube_size,
        y:y+cube_size,
        :
    ]

    patch_pad = np.zeros((128,128,128),dtype=np.float32)

    patch_pad[:patch.shape[0],
              :patch.shape[1],
              :patch.shape[2]] = patch

    return patch_pad

for i in range(100):

    patch = get_patch(volume)

    x, faultseg_mask, faultgan_mask = infer_both_models(patch)

    print(f"\nPatch {i+1}")
    print("FaultSeg ratio :", faultseg_mask.mean())
    print("FaultGAN ratio :", faultgan_mask.mean())

    compare_faultseg_faultgan(
        x,
        faultseg_mask,
        faultgan_mask,
        title=f"Patch {i+1} : FaultSeg vs FaultGAN"
    )

In [ ]:
def get_patch(volume):

    D,H,W = volume.shape

    cube_size = 128

    z = np.random.randint(0, D-cube_size+1)
    y = np.random.randint(0, H-cube_size+1)

    patch = volume[
        z:z+cube_size,
        y:y+cube_size,
        :
    ]

    patch_pad = np.zeros((128,128,128),dtype=np.float32)

    patch_pad[:patch.shape[0],
              :patch.shape[1],
              :patch.shape[2]] = patch

    return patch_pad

for i in range(100):

    patch = get_patch(volume)

    x, faultseg_mask, faultgan_mask = infer_both_models(patch)

    print(f"\nPatch {i+1}")
    print("FaultSeg ratio :", faultseg_mask.mean())
    print("FaultGAN ratio :", faultgan_mask.mean())

    compare_faultseg_faultgan(
        x,
        faultseg_mask,
        faultgan_mask,
        title=f"Patch {i+1} : FaultSeg vs FaultGAN"
    )